# Entrenamiento en la nube (Kaggle / Colab)

Ejecuta los experimentos de la lista de punta a punta: clona el repositorio, descarga y prepara los datos, entrena, evalúa y empaqueta los resultados (`models/` y `experiments/`) en un zip por experimento.

**Kaggle:** si algún experimento usa Toloka (`toloka` o `combined`), adjuntar ese dataset con *Add Input* (buscar "yandextoloka water meters"; una ejecución en segundo plano no puede adjuntarlo sola). Activar *GPU* e *Internet* en *Settings*, cargar el Secret `ROBOFLOW_API_KEY` (*Add-ons → Secrets*) y ejecutar con *Save Version → Save & Run All* para que corra en segundo plano.

**Colab:** *Entorno de ejecución → GPU* y cargar el Secret `ROBOFLOW_API_KEY`.

In [ ]:
# Experimentos a ejecutar, en orden (ver experiments/README.md).
# trainer: "yolo" (Ultralytics, por defecto), "maskrcnn" (torchvision) o "eval_coco"
#          (solo evalúa con métricas COCO un modelo ya versionado en models/).
# task: "segment" (etapa 1, visor) o "detect" (etapa 2, dígitos).
# dataset: "toloka" (solo Toloka), "combined" (Toloka + yfzw4 con visor derivado)
#          o "digits" (dígitos de yfzw4 sobre el visor rectificado).
# fliplr: probabilidad de espejado horizontal; 0 para dígitos (un 2 espejado parece un 5).
# Cada experimento se empaqueta al terminar, así un corte por tiempo no pierde los anteriores:
# por eso van primero los rápidos y al final Mask R-CNN, que es el más largo.
EXPERIMENTS = [
    # B con la misma métrica COCO que Mask R-CNN (comparación justa entre arquitecturas)
    {"name": "toloka_yfzw4_yolov8n_seg", "trainer": "eval_coco", "task": "segment", "dataset": "combined",
     "imgsz": 640},
    # D2: YOLO11n contra YOLOv8n (D1) en dígitos, mismos hiperparámetros
    {"name": "digits_yolo11n", "task": "detect", "dataset": "digits", "model": "yolo11n.pt",
     "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},
    # S4: Mask R-CNN (dos etapas) contra YOLOv8n-seg (B), mismo dataset y tamaño de entrada
    {"name": "toloka_yfzw4_maskrcnn", "trainer": "maskrcnn", "task": "segment", "dataset": "combined",
     "epochs": 10, "batch": 4, "lr": 0.005},
]

# Ya ejecutados (resultados en experiments/README.md):
# {"name": "toloka_yfzw4_yolov8n_seg", "task": "segment", "dataset": "combined", "model": "yolov8n-seg.pt",
#  "imgsz": 640, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},       # B
# {"name": "toloka_yfzw4_yolov8n_seg_1024", "task": "segment", "dataset": "combined", "model": "yolov8n-seg.pt",
#  "imgsz": 1024, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},      # C
# {"name": "digits_yolov8n", "task": "detect", "dataset": "digits", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D1

In [ ]:
# Configuración del entorno: Kaggle, Colab o local.
import os
import subprocess
import sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/martinlacheski/ceia-vpc-2.git"


def run(cmd: list[str]) -> None:
    """Ejecuta un comando mostrando su salida y falla si termina con error."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if IN_KAGGLE or IN_COLAB:
    os.chdir("/kaggle/working" if IN_KAGGLE else "/content")
    if not Path("ceia-vpc-2").exists():
        run(["git", "clone", "-q", REPO_URL])
    os.chdir("ceia-vpc-2")
    # Solo lo necesario para entrenar: PyTorch y torchvision con CUDA ya vienen instalados.
    run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "roboflow", "kagglehub", "opencv-python-headless",
                 "pycocotools"])

    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        else:
            from google.colab import userdata
            os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception as exc:
        print("Aviso: no se pudo leer el Secret ROBOFLOW_API_KEY:", exc)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import torch
print("Directorio:", Path.cwd(), "| Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")

In [ ]:
# Descarga y preparación de los datos (solo los que piden los experimentos).
datasets = {e["dataset"] for e in EXPERIMENTS}
needs_toloka = bool(datasets & {"toloka", "combined"})
needs_yfzw4 = bool(datasets & {"combined", "digits"})
if needs_toloka and needs_yfzw4:
    run([sys.executable, "scripts/download_data.py"])
elif needs_toloka:
    run([sys.executable, "scripts/download_data.py", "--only", "toloka"])
else:
    run([sys.executable, "scripts/download_data.py", "--only", "roboflow"])

if needs_toloka:
    run([sys.executable, "scripts/prepare_toloka.py"])
if "combined" in datasets:
    run([sys.executable, "scripts/prepare_seg_combined.py"])
if "digits" in datasets:
    run([sys.executable, "scripts/prepare_digits.py"])

DATA = {
    "toloka": "data/processed/toloka_seg/data.yaml",
    "combined": "data/processed/seg_toloka_yfzw4/data.yaml",
    "digits": "data/processed/digits_yfzw4/data.yaml",
}

In [ ]:
# Entrenamiento, evaluación y empaquetado de cada experimento.
import json
import shutil

out_dir = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()


def package(name: str, weights: list[str]) -> str:
    """Arma <name>.zip con los pesos y experiments/<name>/ (en Kaggle: pestaña Output)."""
    staging = Path("runs/package") / name
    shutil.rmtree(staging, ignore_errors=True)
    (staging / "models").mkdir(parents=True)
    for path in weights:
        shutil.copy2(path, staging / "models")
    shutil.copytree(f"experiments/{name}", staging / "experiments" / name)
    return shutil.make_archive(str(out_dir / name), "zip", root_dir=staging)


for exp in EXPERIMENTS:
    print(f"\n===== {exp['name']} =====")
    trainer = exp.get("trainer", "yolo")
    if trainer == "eval_coco":
        run([sys.executable, "scripts/evaluate_seg_coco.py", "--arch", "yolo",
             "--weights", f"models/{exp['name']}.pt", "--imgsz", str(exp["imgsz"])])
        print("Resultados:", package(exp["name"], []))
        continue
    if trainer == "maskrcnn":
        root = str(Path(DATA[exp["dataset"]]).parent)
        run([sys.executable, "scripts/train_maskrcnn.py", "--data", root, "--name", exp["name"],
             "--epochs", str(exp["epochs"]), "--batch", str(exp["batch"]), "--lr", str(exp["lr"])])
        run([sys.executable, "scripts/evaluate_seg_coco.py", "--arch", "maskrcnn",
             "--weights", f"models/{exp['name']}.pth"])
        print("Resultados:", package(exp["name"], [f"models/{exp['name']}.pth"]))
        continue

    run([sys.executable, "scripts/train_yolo.py", "--task", exp["task"], "--model", exp["model"],
         "--data", DATA[exp["dataset"]], "--name", exp["name"], "--imgsz", str(exp["imgsz"]),
         "--epochs", str(exp["epochs"]), "--patience", str(exp["patience"]), "--batch", str(exp["batch"]),
         "--fliplr", str(exp["fliplr"])])
    if exp["task"] == "segment":
        run([sys.executable, "scripts/evaluate_seg.py", "--model", f"models/{exp['name']}.pt",
             "--imgsz", str(exp["imgsz"])])
    else:
        # Dígitos: métricas sobre el split de prueba de yfzw4, globales y por clase.
        from ultralytics import YOLO
        res = YOLO(f"models/{exp['name']}.pt").val(data=DATA[exp["dataset"]], split="test",
                                                   imgsz=exp["imgsz"], verbose=False)
        evaluation = {
            "test": {"map50": round(float(res.box.map50), 4), "map50_95": round(float(res.box.map), 4),
                     "precision": round(float(res.box.mp), 4), "recall": round(float(res.box.mr), 4)},
            "map50_95_por_clase": {res.names[int(c)]: round(float(res.box.maps[int(c)]), 4)
                                    for c in res.box.ap_class_index},
            "imgsz": exp["imgsz"],
        }
        Path(f"experiments/{exp['name']}/eval.json").write_text(json.dumps(evaluation, indent=2))
        print(json.dumps(evaluation, indent=2))
    print("Resultados:", package(exp["name"], [f"models/{exp['name']}.pt"]))